**Import Libraries**

In [3]:
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocessing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Models
from sklearn.linear_model import LogisticRegression
!pip install catboost
from catboost import CatBoostClassifier

# Evaluation
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    roc_auc_score
)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 7.3 MB/s eta 0:00:00


**Load Dataset**

In [8]:
df = pd.read_csv("application_train.csv")

print(df.head())

   SK_ID_CURR  TARGET NAME_CONTRACT_TYPE CODE_GENDER FLAG_OWN_CAR  \
0      100002       1         Cash loans           M            N   
1      100003       0         Cash loans           F            N   
2      100004       0    Revolving loans           M            Y   
3      100006       0         Cash loans           F            N   
4      100007       0         Cash loans           M            N   

  FLAG_OWN_REALTY  CNT_CHILDREN  AMT_INCOME_TOTAL  AMT_CREDIT  AMT_ANNUITY  \
0               Y             0          202500.0    406597.5      24700.5   
1               N             0          270000.0   1293502.5      35698.5   
2               Y             0           67500.0    135000.0       6750.0   
3               Y             0          135000.0    312682.5      29686.5   
4               Y             0          121500.0    513000.0      21865.5   

   ...  FLAG_DOCUMENT_18 FLAG_DOCUMENT_19 FLAG_DOCUMENT_20 FLAG_DOCUMENT_21  \
0  ...               0.0             

**Check Missing Values**

In [9]:
print(df.isnull().sum())

SK_ID_CURR                       0
TARGET                           0
NAME_CONTRACT_TYPE               0
CODE_GENDER                      0
FLAG_OWN_CAR                     0
                              ... 
AMT_REQ_CREDIT_BUREAU_DAY     2085
AMT_REQ_CREDIT_BUREAU_WEEK    2085
AMT_REQ_CREDIT_BUREAU_MON     2085
AMT_REQ_CREDIT_BUREAU_QRT     2085
AMT_REQ_CREDIT_BUREAU_YEAR    2085
Length: 122, dtype: int64


**Handle Missing Values**

In [10]:
# Fill numeric columns
df.fillna(df.median(numeric_only=True), inplace=True)

# Fill categorical columns
for col in df.select_dtypes(include='object').columns:
    df[col].fillna(df[col].mode()[0], inplace=True)

/tmp/ipykernel_4321/1802935255.py:6: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(df[col].mode()[0], inplace=True)


**Encode Categorical Data**

In [11]:
label_encoder = LabelEncoder()

for col in df.select_dtypes(include='object').columns:
    df[col] = label_encoder.fit_transform(df[col])

**Define Features and Target**

In [12]:
 X = df.drop('TARGET', axis=1)

y = df['TARGET']

**Split Dataset**

In [13]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

**Train Logistic Regression**

In [14]:
log_model = LogisticRegression(max_iter=1000)

log_model.fit(X_train, y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression(max_iter=1000)

**Train CatBoost Model**

In [15]:
cat_model = CatBoostClassifier(verbose=0)

cat_model.fit(X_train, y_train)

CatBoostClassifier(verbose=0)

**Make Predictions**

In [16]:
 pred_probs = cat_model.predict_proba(X_test)[:,1]

**Define Costs**

In [17]:
FP_cost = 1000
FN_cost = 10000

**Adjust Decision Threshold**

In [18]:
threshold = 0.3

predictions = (pred_probs >= threshold).astype(int)

**Confusion Matrix**

In [19]:
cm = confusion_matrix(y_test, predictions)

print(cm)

[[2807   50]
 [ 229   19]]


**Calculate Business Cost**

In [20]:
TN, FP, FN, TP = cm.ravel()

total_cost = (FP * FP_cost) + (FN * FN_cost)

print("Total Business Cost:", total_cost)

Total Business Cost: 2340000


**Find Best Threshold**

In [21]:
 thresholds = np.arange(0.1, 1.0, 0.1)

costs = []

for t in thresholds:

    preds = (pred_probs >= t).astype(int)

    cm = confusion_matrix(y_test, preds)

    TN, FP, FN, TP = cm.ravel()

    cost = (FP * FP_cost) + (FN * FN_cost)

    costs.append(cost)

best_threshold = thresholds[np.argmin(costs)]

print("Best Threshold:", best_threshold)

Best Threshold: 0.1


**Feature Importance**

In [22]:
importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': cat_model.feature_importances_
})

importance = importance.sort_values(
    by='Importance',
    ascending=False
)

print(importance.head(10))

                       Feature  Importance
42                EXT_SOURCE_3    9.678125
41                EXT_SOURCE_2    7.208995
94      DAYS_LAST_PHONE_CHANGE    4.006909
8                  AMT_ANNUITY    4.005256
16                  DAYS_BIRTH    3.923737
40                EXT_SOURCE_1    3.923704
18           DAYS_REGISTRATION    3.701012
15  REGION_POPULATION_RELATIVE    3.578409
19             DAYS_ID_PUBLISH    3.483864
17               DAYS_EMPLOYED    3.069639


In [23]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive
